#### QUERY `GIZMO.BRONZE.CUSTOMERS_DELTA`
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: BRONZE
3. VIEW NAME: CUSTOMERS_DELTA

In [0]:
%python
customers_df = spark.table("GIZMO.BRONZE.CUSTOMERS_DELTA").distinct().orderBy('customer_id').filter('customer_id IS NOT NULL')
display(customers_df)

#### DEFINE UDF TO EXTRACT FIRST NAME FROM CUSTOMER_NAME

In [0]:
def extract_first_name(customer_name):
    return customer_name.split(' ')[0] if customer_name else None

#### DEFINE UDF TO EXTRACT LAST NAME FROM CUSTOMER_NAME

In [0]:
def extract_last_name(customer_name):
    if customer_name and ' ' in customer_name:
        return customer_name.split(' ', 1)[1]
    return None

#### REGISTER `EXTRACT_FIRST_NAME`, `EXTRACT_LAST_NAME` AS UDF

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Register the function as a UDF
first_name_udf = udf(extract_first_name, StringType())
last_name_udf = udf(extract_last_name, StringType())

In [0]:
customers_name_extract_df = (customers_df.select('customer_id', 'customer_name','date_of_birth', 'email', 'member_since', 
                                            'telephone', 'created_timestamp', 'file_name', 'file_path')
                                      .withColumn('first_name', first_name_udf('customer_name'))
                                      .withColumn('last_name', last_name_udf('customer_name'))
                                      .drop('customer_name')
                                      )
display(customers_name_extract_df)

#### SELECT REQUIRED COLUMNS
1. **`customer_id`, `first_name`, `last_name`, `date_of_birth`, `email`, `member_since`, `telephone`, `created_timestamp`, `file_name`, `file_path`**
2. SORT THE DATAFRAME BASED ON `CUSTOMER_ID` IN ASCENDING
3. FILTER OUT `CUSTOMER_ID` NOT NULL


In [0]:
customers_selected_cols_df = customers_name_extract_df.select('customer_id', 'first_name', 'last_name', 'date_of_birth', 'email', 'member_since', 
                                            'telephone', 'created_timestamp', 'file_name', 'file_path')
display(customers_selected_cols_df)

#### NEED TO LATEST CUSTOMERS BASED ON CREATED_TIMESTAMP

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

windowSpec = Window.partitionBy("customer_id").orderBy(col("created_timestamp").desc())

ranked_customer_df = customers_selected_cols_df.withColumn("rank", rank().over(windowSpec)).filter('rank = 1')
display(ranked_customer_df)

#### WRITE CUSTOMERS TO DELTA FORMAT

In [0]:
ranked_customer_df.writeTo('gizmo.silver.customers_delta').createOrReplace()

#### QUERY AND VALIDATE `GIZMO.SILVER.CUSTOMERS`

In [0]:
%sql
SELECT * FROM GIZMO.SILVER.CUSTOMERS_DELTA ORDER BY 1;

#### VALIDATE RECORD COUNT FROM `GIZMO.SILVER.CUSTOMERS_DELTA`;

In [0]:
%sql
SELECT COUNT(*) AS RECORD_CNT
FROM
GIZMO.SILVER.CUSTOMERS_DELTA;

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS LOADED INTO GIZMO.SILVER.CUSTOMERS")